# Load the IBM TabFormer credit-card dataset

This notebook downloads IBM's synthetic credit-card transactions and writes them to a managed Delta table. The download is staged temporarily in a Unity Catalog volume so Spark workers can read the extracted CSV.

In [ ]:
%pip install pyyaml

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
import os
import shutil
import tarfile
from urllib.request import Request, urlopen

import yaml
from pyspark.sql.types import (
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

TABFORMER_URL = (
    "https://github.com/IBM/TabFormer/raw/main/data/credit_card/transactions.tgz"
)

CONFIG_FILE = "config.yaml"


def resolve_config_path() -> Path:
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent / CONFIG_FILE)
    candidates.append(Path.cwd() / CONFIG_FILE)

    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        notebook_path = context.notebookPath().get()
    except Exception:
        pass
    else:
        candidates.append(
            Path("/Workspace")
            / notebook_path.lstrip("/").rsplit("/", 1)[0]
            / CONFIG_FILE
        )

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {CONFIG_FILE}; searched: {searched}")


config_path = resolve_config_path()
with config_path.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

if not isinstance(config, dict):
    raise ValueError(f"Expected a YAML mapping in {config_path}")


def required_mapping(mapping: dict, key: str) -> dict:
    value = mapping.get(key)
    if not isinstance(value, dict):
        raise ValueError(f"Config value {key} must be a mapping")
    return value


def required_string(mapping: dict, key: str) -> str:
    value = str(mapping.get(key, "")).strip()
    if not value:
        raise ValueError(f"Missing required config value: {key}")
    return value


def quote_identifier(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"


data_config = required_mapping(config, "data")
catalog = required_string(data_config, "catalog")
schema = required_string(data_config, "schema")
table = required_string(data_config, "table")
staging_volume = required_string(data_config, "staging_volume")

schema_name = ".".join(map(quote_identifier, (catalog, schema)))
table_name = ".".join(map(quote_identifier, (catalog, schema, table)))
volume_name = ".".join(
    map(quote_identifier, (catalog, schema, staging_volume))
)
print(f"Using configuration from {config_path}")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")

staging_dir = (
    Path("/Volumes")
    / catalog
    / schema
    / staging_volume
    / "_ibm_tabformer_download"
)
archive_path = staging_dir / "transactions.tgz"
extract_dir = staging_dir / "extracted"
staging_dir.mkdir(parents=True, exist_ok=True)

if not archive_path.exists():
    temporary_path = archive_path.with_suffix(".tgz.download")
    request = Request(
        TABFORMER_URL,
        headers={"User-Agent": "Databricks-TabFormer-Setup/1.0"},
    )
    print(f"Downloading IBM TabFormer from {TABFORMER_URL}")
    with urlopen(request, timeout=60) as response:
        with temporary_path.open("wb") as output_file:
            shutil.copyfileobj(response, output_file, length=16 * 1024 * 1024)
    temporary_path.replace(archive_path)
else:
    print(f"Using staged archive {archive_path}")

with archive_path.open("rb") as archive_file:
    if archive_file.read(2) != b"\x1f\x8b":
        raise ValueError(f"Downloaded file is not a gzip archive: {archive_path}")

In [ ]:
def safe_extract(archive: tarfile.TarFile, destination: Path) -> None:
    destination = destination.resolve()
    for member in archive.getmembers():
        if member.issym() or member.islnk():
            raise ValueError(f"Archive contains a link: {member.name}")
        member_path = (destination / member.name).resolve()
        if os.path.commonpath((destination, member_path)) != str(destination):
            raise ValueError(f"Unsafe path in archive: {member.name}")
    archive.extractall(destination)


csv_files = sorted(extract_dir.rglob("*.csv"))
if not csv_files:
    extract_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path}")
    with tarfile.open(archive_path, "r:gz") as archive:
        safe_extract(archive, extract_dir)
    csv_files = sorted(extract_dir.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {archive_path}")

transactions_csv = max(csv_files, key=lambda path: path.stat().st_size)
print(f"Reading {transactions_csv}")

In [ ]:
source_schema = StructType(
    [
        StructField("User", IntegerType(), True),
        StructField("Card", IntegerType(), True),
        StructField("Year", IntegerType(), True),
        StructField("Month", IntegerType(), True),
        StructField("Day", IntegerType(), True),
        StructField("Time", StringType(), True),
        StructField("Amount", StringType(), True),
        StructField("Use Chip", StringType(), True),
        StructField("Merchant Name", LongType(), True),
        StructField("Merchant City", StringType(), True),
        StructField("Merchant State", StringType(), True),
        StructField("Zip", StringType(), True),
        StructField("MCC", IntegerType(), True),
        StructField("Errors?", StringType(), True),
        StructField("Is Fraud?", StringType(), True),
    ]
)

column_names = [
    "user_id",
    "card_id",
    "year",
    "month",
    "day",
    "time",
    "amount",
    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip_code",
    "mcc",
    "errors",
    "is_fraud",
]

transactions = (
    spark.read.option("header", True)
    .schema(source_schema)
    .csv(transactions_csv.as_posix())
    .toDF(*column_names)
)

In [ ]:
(
    transactions.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

spark.sql(
    f"COMMENT ON TABLE {table_name} IS "
    "'IBM TabFormer synthetic credit-card transactions'"
)

shutil.rmtree(staging_dir)
print(f"Wrote IBM TabFormer transactions to {table_name}")

## Verify the Delta table

In [ ]:
display(spark.table(table_name).limit(10))